# E7 — Инъекция отказов и безопасность (safety-часть Г4)

SINDy-MPC на тест-сезоне 2020 под отказами датчиков/актуаторов (stuck/offset/dead, появляются в середине сезона) — с residual-супервизором и без. Супервизор ловит скачок остатка одношагового прогноза суррогата (онсет отказа / актуаторная несогласованность) и **залипает** в безопасный режим (откат на rule_based, [rollout_mpc_faulty](article_experiment_utils.py)). Полный прогон — run_e7_faults.py + merge_e7.py.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location); CORR, PRICES = ECON["corridors"], ECON["prices"]
RECIPE = dict(feature_variant="physics_no_cross", library_degree=1, optimizer="stlsq", denoise="none")
print("FAST_MODE", FAST_MODE)

FAST_MODE True


## Прогон под отказами (с супервизором и без)

In [2]:
test = pc.test_scenario(); SS = test["start_date"]; s = 0; cfg = pc.cfg_for(test, seed=s)
N = 5 if FAST_MODE else 30; n_train = 7 if FAST_MODE else 30
onset = (N * int(86400 / pc.period)) // 3   # fault appears 1/3 into the season
parts = [U.collect_rule_based_dataset(pc.cfg_for({"year": y, "start_date": f"{y}-03-01", "n_days": n_train}, seed=s), n_days=n_train, prbs_scale=0.3) for y in (2018, 2019)]
b = U.fit_sindy(U.aggregate_trajectories(parts, pc.base_cfg(n_train)), period=float(pc.period), **RECIPE)
ALL = [("t_in_stuck", {"layer": "sensor", "target": "t_in", "type": "stuck", "value": 25.0}),
       ("rh_offset", {"layer": "sensor", "target": "rh", "type": "offset", "value": -25.0}),
       ("uVent_dead", {"layer": "actuator", "target": "uVent", "type": "dead", "value": 0.0}),
       ("uBoil_stuck", {"layer": "actuator", "target": "uBoil", "type": "stuck", "value": 1.0})]
FAULTS = ALL[:2] if FAST_MODE else ALL
rows = []
def add(fault, sup, df):
    m = U.epi_metrics(df, corridors=CORR, prices=PRICES)
    fl = float(df["flagged"].mean()) if "flagged" in df.columns else 0.0
    rows.append({"fault": fault, "supervised": sup, "epi": m["epi"], "viol": m["violation_steps_total"], "flag_frac": fl})
add("none", 0, U.rollout_mpc(b, cfg, N, start_date=SS))
for name, spec in FAULTS:
    f = {**spec, "start_step": onset}
    add(name, 0, U.rollout_mpc_faulty(b, cfg, N, f, start_date=SS, supervisor=False))
    add(name, 1, U.rollout_mpc_faulty(b, cfg, N, f, start_date=SS, supervisor=True, resid_threshold=3.0))
df = pd.DataFrame(rows); print(df.round(3).to_string(index=False))

     fault  supervised   epi  viol  flag_frac
      none           0 0.283   471      0.000
t_in_stuck           0 0.390   562      0.000
t_in_stuck           1 0.401   455      0.283
 rh_offset           0 0.415   594      0.000
 rh_offset           1 0.392   307      0.667


## Таблица деградации + рисунок

In [3]:
bv = float(df[df.fault == "none"]["viol"].iloc[0]); be = float(df[df.fault == "none"]["epi"].iloc[0])
faults = [f for f in df.fault.unique() if f != "none"]
tab = pd.DataFrame([{
    "fault": f,
    "viol_unsup": float(df[(df.fault == f) & (df.supervised == 0)]["viol"].iloc[0]),
    "viol_sup": float(df[(df.fault == f) & (df.supervised == 1)]["viol"].iloc[0]),
    "epi_unsup": float(df[(df.fault == f) & (df.supervised == 0)]["epi"].iloc[0]),
    "epi_sup": float(df[(df.fault == f) & (df.supervised == 1)]["epi"].iloc[0]),
    "flag_frac": float(df[(df.fault == f) & (df.supervised == 1)]["flag_frac"].iloc[0]),
} for f in faults]).sort_values("viol_unsup", ascending=False)
U.save_table(tab, RES / "tables" / "e7_degradation.csv")
fig, ax = plt.subplots(figsize=(9, 4)); x = np.arange(len(tab)); w = 0.38
ax.bar(x - w / 2, tab.viol_unsup, w, label="no supervisor"); ax.bar(x + w / 2, tab.viol_sup, w, label="with supervisor")
ax.axhline(bv, color="k", ls="--", lw=1, label=f"fault-free ({bv:.0f})")
ax.set_xticks(x, tab.fault, rotation=25, ha="right"); ax.set_ylabel("violation steps"); ax.set_title("E7 fault degradation & supervisor"); ax.legend(fontsize=8)
U.save_figure(fig, RES / "figures" / "e7_faults.png"); plt.close(fig)
print(f"fault-free: EPI={be:.3f} viol={bv:.0f}"); display(tab.round(3))

fault-free: EPI=0.283 viol=471


,fault,viol_unsup,viol_sup,epi_unsup,epi_sup,flag_frac
1,rh_offset,594.0,307.0,0.415,0.392,0.667
0,t_in_stuck,562.0,455.0,0.390,0.401,0.283


**Итог E7.** Отказы повышают нарушения коридоров (а stuck-актуатор бьёт и по EPI). Residual-супервизор детектирует и смягчает **онсетные/актуаторные** отказы (t_in stuck, uVent dead — нарушения вниз малой ценой EPI), но **постоянные тонкие смещения датчиков** (rh offset, uBoil stuck) остаются ниже порога — честный предел одношагового детектора; для них нужна кросс-сенсорная консистентность (далее).